# Grad-CAM — DenseNet-121
See where the model looks when it reads an X-ray.

---
## Setup

In [ ]:
%pip install grad-cam -q

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

---
## Step 1 — Load the Trained DenseNet-121

In [ ]:
model = models.densenet121(weights=None)
model.classifier = nn.Linear(model.classifier.in_features, 3)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()
print("Model loaded.")

---
## Step 2 — Put Your Image Path Here

Change the path below to any X-ray image you want.

In [ ]:
# ← Change this to your image path
IMAGE_PATH = r"put your image path here"

image = Image.open(IMAGE_PATH).convert("RGB")
plt.imshow(image)
plt.title("Your Image")
plt.axis("off")
plt.show()

---
## Step 3 — Run Grad-CAM

In [ ]:
CLASSES = ["Emphysema", "Normal", "Other"]

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

image_resized = image.resize((224, 224))
image_np      = np.array(image_resized).astype(np.float32) / 255.0
input_tensor  = transform(image).unsqueeze(0).to(device)

# Prediction
with torch.no_grad():
    outputs = model(input_tensor)
    probs   = torch.softmax(outputs, dim=1)[0]
    pred    = probs.argmax().item()

print(f"Prediction : {CLASSES[pred]}")
print(f"Confidence : {probs[pred].item()*100:.1f}%")

# Grad-CAM
target_layer = [model.features.denseblock4]
with GradCAM(model=model, target_layers=target_layer) as cam:
    grayscale_cam = cam(input_tensor=input_tensor)[0]

result = show_cam_on_image(image_np, grayscale_cam, use_rgb=True)

# Show side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image_resized)
axes[0].set_title("Original", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(result)
axes[1].set_title(f"Grad-CAM — {CLASSES[pred]} ({probs[pred].item()*100:.1f}%)", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()